In [485]:
import os
import plotly.express as px
from sklearn.cluster import DBSCAN
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
import imageio
from scipy.optimize import linear_sum_assignment
from typing import Dict, Tuple, List, Any
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

In [434]:
route = "/home/eder/projects/car-cluster-project/Data/pointclouds"
csvs = os.listdir("/home/eder/projects/car-cluster-project/Data/pointclouds")
csvs_sorted = sorted(csvs, key=lambda x: int(x.split('_')[1:][0].split('.')[0]))
csvs_sorted[:5]

['pointcloud_1727346186_987057480.csv',
 'pointcloud_1727346186_586942416.csv',
 'pointcloud_1727346186_488981170.csv',
 'pointcloud_1727346186_936946076.csv',
 'pointcloud_1727346186_738318465.csv']

In [435]:
# Get first frame
csv_1 = csvs_sorted[0]
frame_1 = pd.read_csv(f"{route}/{csv_1}")

# Apply DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=100)
dbscan.fit(frame_1)
labels = dbscan.labels_

# Visualize
fig_frame_1 = px.scatter_3d(frame_1, x='x', y='y', z='z')
frame_1['cluster'] = labels
fig_frame_1_clusters = px.scatter_3d(frame_1, x='x', y='y', z='z', color='cluster')
fig_frame_1_clusters.update_layout(scene=dict(zaxis=dict(range=[-20, 20])))
fig_frame_1_clusters.update_layout(scene=dict(xaxis=dict(range=[-70, 20]), yaxis=dict(range=[-20, 20])))
fig_frame_1_clusters.update_traces(marker=dict(size=2))
fig_frame_1_clusters.show()

We take a representative sample of the video

In [436]:
csv_sample = [csvs[i] for i in range(0, 369, 40 )]

csv_sample

['pointcloud_1727346204_841877402.csv',
 'pointcloud_1727346188_37776056.csv',
 'pointcloud_1727346194_190934310.csv',
 'pointcloud_1727346204_989042406.csv',
 'pointcloud_1727346202_941074715.csv',
 'pointcloud_1727346205_40307462.csv',
 'pointcloud_1727346191_589816812.csv',
 'pointcloud_1727346196_96805998.csv',
 'pointcloud_1727346204_891963987.csv',
 'pointcloud_1727346188_838958503.csv']

In [437]:
'''for csv in csv_sample:
    print(f"Processing {csv}...")
    dbscan = DBSCAN(eps=2, min_samples=10, n_jobs=-1)
    frame_1 = pd.read_csv(f"{route}/{csv}")
    dbscan.fit(frame_1)
    labels = dbscan.labels_
    frame_1['cluster'] = labels
    # frame_1 = frame_1[frame_1['cluster'] != -1]
    fig_frame_1_clusters = px.scatter_3d(frame_1, x='x', y='y', z='z', color='cluster')
    fig_frame_1_clusters.update_layout(scene=dict(zaxis=dict(range=[-20, 20])))
    fig_frame_1_clusters.update_layout(scene=dict(xaxis=dict(range=[-70, 20]), yaxis=dict(range=[-20, 20])))
    fig_frame_1_clusters.update_traces(marker=dict(size=2))
    fig_frame_1_clusters.show()'''

'for csv in csv_sample:\n    print(f"Processing {csv}...")\n    dbscan = DBSCAN(eps=2, min_samples=10, n_jobs=-1)\n    frame_1 = pd.read_csv(f"{route}/{csv}")\n    dbscan.fit(frame_1)\n    labels = dbscan.labels_\n    frame_1[\'cluster\'] = labels\n    # frame_1 = frame_1[frame_1[\'cluster\'] != -1]\n    fig_frame_1_clusters = px.scatter_3d(frame_1, x=\'x\', y=\'y\', z=\'z\', color=\'cluster\')\n    fig_frame_1_clusters.update_layout(scene=dict(zaxis=dict(range=[-20, 20])))\n    fig_frame_1_clusters.update_layout(scene=dict(xaxis=dict(range=[-70, 20]), yaxis=dict(range=[-20, 20])))\n    fig_frame_1_clusters.update_traces(marker=dict(size=2))\n    fig_frame_1_clusters.show()'

In this part we have to fine tune the DBSAN model to get it work in all the samples we are working with.

In [438]:
def compute_median_per_cluster(labels, df: pd.DataFrame):
    # assumes df has 'x','y','z' columns
    med = {}
    # work on a temporary frame; do not modify df
    tmp = pd.DataFrame({'cluster': labels, 'x': df['x'].to_numpy(),
                        'y': df['y'].to_numpy(), 'z': df['z'].to_numpy()})
    tmp = tmp[tmp['cluster'] != -1]
    if tmp.empty:
        return med
    g = tmp.groupby('cluster', as_index=False)[['x','y','z']].median()
    for _, row in g.iterrows():
        med[int(row['cluster'])] = (float(row['x']), float(row['y']), float(row['z']))
    return med


In [ ]:
def pair_match_min_distance(
    A: Dict[Any, Tuple[float, float, float]],   # previous {keyA: (x,y,z)}
    B: Dict[Any, Tuple[float, float, float]],   # current  {keyB: (x,y,z)}
    max_distance: float | None = None,
    allow_unmatched: bool = True,
    skip_cost: float | None = None,
):
    """
    Hungarian match between dicts of 3D points.
    Returns:
        pairs: [(keyA, keyB, distance)]
        unmatched_A: [keyA]
        unmatched_B: [keyB]
    """

    keysA = list(A.keys())
    keysB = list(B.keys())

    ArrA = np.asarray([A[k] for k in keysA], dtype=float)  # (n,3)
    ArrB = np.asarray([B[k] for k in keysB], dtype=float)  # (m,3)

    # Pairwise distances with broadcasting
    diff = ArrA[:, None, :] - ArrB[None, :, :]             # (n,m,3)
    D = np.linalg.norm(diff, axis=2)                       # (n,m)

    # Replace invalids with a huge cost
    BIG = 1e12
    finite_mask = np.isfinite(D)
    if not finite_mask.any():
        # nothing finite -> cannot match
        return [], keysA, keysB
    max_finite = D[finite_mask].max()
    D = np.where(finite_mask, D, BIG)

    # If we do not allow unmatched, do rectangular assignment
    if not allow_unmatched:
        r, c = linear_sum_assignment(D)
        pairs = [(keysA[i], keysB[j], float(D[i, j])) for i, j in zip(r, c)]
        unmatched_A = [keysA[i] for i in sorted(set(range(D.shape[0])) - set(r))]
        unmatched_B = [keysB[j] for j in sorted(set(range(D.shape[1])) - set(c))]
        return pairs, unmatched_A, unmatched_B

    # Allow unmatched: build square cost matrix with dummies
    pad = (max_finite + 1.0) if (skip_cost is None) else float(skip_cost)
    # ensure pad > any real distance so real matches are preferred
    if pad <= max_finite:
        pad = max_finite + 1.0

    n, m = D.shape
    size = n + m
    C = np.full((size, size), BIG, dtype=float)
    # real distances
    C[:n, :m] = D
    # A -> dummy columns (leave A[i] unmatched)
    C[:n, m:m+n] = pad
    # dummy rows -> B (leave B[j] unmatched)
    C[n:n+m, :m] = pad
    # bottom-right remains BIG (dummy-dummy forbidden)

    r, c = linear_sum_assignment(C)

    pairs_idx: List[Tuple[int, int]] = []
    unmatched_A_idx = set(range(n))
    unmatched_B_idx = set(range(m))

    for i, j in zip(r, c):
        if i < n and j < m:
            pairs_idx.append((i, j))
            unmatched_A_idx.discard(i)
            unmatched_B_idx.discard(j)
        # other cases are matches to dummies → remain unmatched

    # Fallback: if somehow all matched to dummies, try rectangular
    if not pairs_idx and (n > 0 and m > 0):
        rr, cc = linear_sum_assignment(D)
        pairs_idx = list(zip(rr, cc))
        unmatched_A_idx = set(range(n)) - set(rr)
        unmatched_B_idx = set(range(m)) - set(cc)

    pairs = [(keysA[i], keysB[j], float(D[i, j])) for i, j in pairs_idx]
    unmatched_A = [keysA[i] for i in sorted(unmatched_A_idx)]
    unmatched_B = [keysB[j] for j in sorted(unmatched_B_idx)]

    # Remove pairs with big distance (invalid matches)
    valid_pairs = []
    for (ka, kb, dist) in pairs:
        if max_distance > dist:
            valid_pairs.append((ka, kb, dist))
        else:
            unmatched_A.append(ka)
            unmatched_B.append(kb)
    pairs = valid_pairs
    return pairs, unmatched_A, unmatched_B

In [440]:
# Counter iterator
def id():
    i = 0
    while True:
        yield i
        i += 1

In [441]:
def assign_matches(pairs, unmatchedB, labels, current_medians, id_colors, colors):
    good_current_medians = {}
    good_labels = labels.copy()
    for j, k, _ in pairs:
        good_labels[labels == k] = j
        good_current_medians[j] = current_medians.pop(k)

    for cluster in unmatchedB:
        id = next(id_counter)
        good_labels[labels ==cluster] = id
        id_colors[id] = colors[id]
        good_current_medians[id] = current_medians[cluster]

    return good_labels, good_current_medians

In [442]:
def new_objects(new_objects, id_colors, colors, labels):
    for cluster in new_objects:
        id = next(id_counter)
        labels[cluster] = id
        id_colors[id] = colors[id]
    return labels

In [ ]:
def plot_state():
    fig = plt.figure(figsize=(10, 5), dpi=120)
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim([-70, 20])
    ax.set_ylim([-20, 20])
    ax.set_zlim([-20, 20])
    ax.set_box_aspect([4, 1, 1])  # Different aspect ratio
    ax.view_init(elev=10., azim=120)
    
    # base artists we’ll update
    scat_points = ax.scatter([], [], [], s=3)           # all points
    scat_medians = ax.scatter([], [], [], s=40, marker='x')  # cluster medians
    return fig, ax, scat_points, scat_medians

In [444]:
def track_disappeared(medians_previous_frame, current_medians, disappeared_medians, id_colors):
    for k in medians_previous_frame.keys():
        if k not in current_medians:
            if k not in disappeared_medians:
                disappeared_medians[k] = 1
                current_medians[k] = medians_previous_frame[k]
            else:
                disappeared_medians[k] = disappeared_medians.get(k, 0) + 1
                if disappeared_medians[k] > 4:
                    id_colors.pop(k, None)
                    disappeared_medians.pop(k, None)
                else:
                    current_medians[k] = medians_previous_frame[k]

In [ ]:
def setup_figure():
    fig = plt.figure(figsize=(16, 8), dpi=200)
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim([-70, 10])
    ax.set_ylim([-5, 5])
    ax.set_zlim([-5, 5])
    ax.set_box_aspect([4, 1, 1])
    ax.view_init(elev=30., azim=-90)

    #distance (overall zoom)
    ax.dist = 5  # default ~10; smaller -> closer, larger -> farther

    # base artists we’ll update
    scat_points = ax.scatter([], [], [], s=3)           # all points
    scat_medians = ax.scatter([], [], [], s=40, marker='x')  # cluster medians
    return fig, ax, scat_points, scat_medians

def _rgba(color, alpha):
    r,g,b,_ = mcolors.to_rgba(color)
    return (r, g, b, float(alpha))

def compute_padded_aabbs_by_id(
    df: pd.DataFrame,
    labels: np.ndarray,
    pad_rel: float = 0.10,     # 10% of each extent
    pad_abs: float = 0.5,      # or at least 0.5 units padding
    min_size: float = 1.0,     # ensure box not thinner than this (per axis)
    min_pts: int = 10          # ignore tiny clusters
):
    """
    Return {track_id: ((xmin,xmax),(ymin,ymax),(zmin,zmax))} with padding.
    """
    out = {}
    arr = df[['x','y','z']].to_numpy()
    for tid in np.unique(labels):
        mask = (labels == tid)
        if np.count_nonzero(mask) < min_pts:
            continue
        pts = arr[mask]
        mins = pts.min(axis=0)  # [xmin,ymin,zmin]
        maxs = pts.max(axis=0)  # [xmax,ymax,zmax]
        span = np.maximum(maxs - mins, 1e-6)

        # padding per-axis: max(relative, absolute)
        pad = np.maximum(pad_rel * span, pad_abs)

        # enforce minimum size (center the expansion)
        desired = np.maximum(span + 2*pad, min_size)
        center = (mins + maxs) * 0.5
        half = desired * 0.5

        bb_min = center - half
        bb_max = center + half
        out[int(tid)] = ((bb_min[0], bb_max[0]), (bb_min[1], bb_max[1]), (bb_min[2], bb_max[2]))
    return out

def smooth_aabbs(prev: dict, curr: dict, alpha: float = 0.6):
    """
    Exponential smoothing of boxes to reduce flicker:
    new = alpha*curr + (1-alpha)*prev  (on mins/maxs separately)
    """
    smoothed = {}
    for tid, bb in curr.items():
        if tid in prev:
            (xmin,xmax),(ymin,ymax),(zmin,zmax) = bb
            (pxmin,pxmax),(pymin,pymax),(pzmin,pzmax) = prev[tid]
            sxmin = alpha*xmin + (1-alpha)*pxmin
            sxmax = alpha*xmax + (1-alpha)*pxmax
            symin = alpha*ymin + (1-alpha)*pymin
            symax = alpha*ymax + (1-alpha)*pymax
            szmin = alpha*zmin + (1-alpha)*pzmin
            szmax = alpha*zmax + (1-alpha)*pzmax
            smoothed[tid] = ((sxmin,sxmax),(symin,symax),(szmin,szmax))
        else:
            smoothed[tid] = bb
    return smoothed

def _aabb_faces(bbox):
    """Return 6 quad faces from ((xmin,xmax),(ymin,ymax),(zmin,zmax))."""
    (xmin,xmax),(ymin,ymax),(zmin,zmax) = bbox
    c = np.array([
        [xmin,ymin,zmin],[xmax,ymin,zmin],[xmax,ymax,zmin],[xmin,ymax,zmin],
        [xmin,ymin,zmax],[xmax,ymin,zmax],[xmax,ymax,zmax],[xmin,ymax,zmax],
    ])
    F = [
        [0,1,2,3], [4,5,6,7],     # bottom, top
        [0,1,5,4], [1,2,6,5],     # sides
        [2,3,7,6], [3,0,4,7],
    ]
    return [c[idx] for idx in F]

def update_aabbs(
    ax,
    aabbs_by_id: dict,         # {tid: ((xmin,xmax),...)}
    id_colors: dict,           # {tid: color}
    aabb_artists: dict,        # {tid: Poly3DCollection}
    face_alpha: float = 0.12,
    edge_alpha: float = 0.95,
    edge_width: float = 3.0
):
    # remove stale
    for tid in list(aabb_artists.keys()):
        if tid not in aabbs_by_id:
            aabb_artists[tid].remove()
            del aabb_artists[tid]
    # upsert
    for tid, bb in aabbs_by_id.items():
        faces = _aabb_faces(bb)
        base = id_colors.get(int(tid), "#000000")
        fc = _rgba(base, face_alpha)
        ec = _rgba(base, edge_alpha)
        if tid in aabb_artists:
            poly = aabb_artists[tid]
            poly.set_verts(faces)
            poly.set_facecolor(fc)
            poly.set_edgecolor(ec)
            poly.set_linewidth(edge_width)
        else:
            poly = Poly3DCollection(faces, facecolors=fc, edgecolors=ec, linewidths=edge_width)
            try: poly.set_depthshade(False)
            except Exception: pass
            ax.add_collection3d(poly)
            aabb_artists[tid] = poly


def plot_tracking(
    scat_points,
    scat_medians,
    df: pd.DataFrame,
    labels: np.ndarray,          # per-point *track_id* (after your assign_matches)
    id_colors: dict,             # {track_id: "#RRGGBB" or rgba}
    current_medians: dict | None = None,  # {track_id: (x,y,z)} if you want to plot medians
):

    # ---- coords ----
    arr = df.select_dtypes(include=[np.number]).iloc[:, :3].to_numpy()
    x, y, z = arr[:, 0], arr[:, 1], arr[:, 2]

    # ---- mask out noise (keep arrays aligned) ----
    point_colors = np.empty((len(labels), 4), dtype=float)
    for id in current_medians.keys():
        color = mcolors.to_rgba(id_colors.get(int(id)))
        point_colors[labels == id] = color

    # ---- update point scatter ----
    scat_points._offsets3d = (x, y, z)
    scat_points.set_facecolors(point_colors)
    scat_points.set_edgecolors(point_colors)

def plot_medians(
    scat_points,      # ignored here (we only update medians)
    scat_medians,
    df: pd.DataFrame,           # not used here, kept for signature compatibility
    labels: np.ndarray,         # not used here
    id_colors: dict,            # {track_id: "#RRGGBB" or rgba}
    current_medians: dict | None = None,  # {track_id: (x,y,z)}
    median_size: float = 300.0,           # points^2 (Matplotlib convention)
    median_marker_edgewidth: float = 2.0, # thicker edges
    use_facecolor: bool = False,          # True if you use marker='o'
):
    if not current_medians:
        scat_medians._offsets3d = ([], [], [])
        scat_medians.set_sizes([])
        return

    # Keep order stable
    tids = list(current_medians.keys())
    meds = np.asarray([current_medians[t] for t in tids], dtype=float)
    mx, my, mz = meds[:, 0], meds[:, 1], meds[:, 2]

    # Update positions
    scat_medians._offsets3d = (mx, my, mz)

    # Set sizes (one per point)
    sizes = np.full(len(tids), float(median_size), dtype=float)
    scat_medians.set_sizes(sizes)

    # Colors: for marker='x', edgecolors matter; facecolors are ignored
    edge_cols = [mcolors.to_rgba(id_colors.get(int(t), "#000000")) for t in tids]
    scat_medians.set_edgecolors(edge_cols)
    scat_medians.set_linewidths(median_marker_edgewidth)

    # If your marker is 'o' (filled), use facecolors:
    if use_facecolor:
        scat_medians.set_facecolors(edge_cols)
    else:
        # For 'x', keep face fully transparent (but NOT the edges!)
        scat_medians.set_facecolors([(0, 0, 0, 0)] * len(tids))

    # Optional: avoid depth shading dimming the color
    try:
        scat_medians.set_depthshade(False)
    except Exception:
        pass

def make_video(route, csv_files, out_path, fps=20):
    # FFMPEG writer (no pyav)
    writer = imageio.get_writer(
        out_path,
        fps=fps,
        codec="libx264",
        format="FFMPEG",
        pixelformat="yuv420p",
        macro_block_size=None,  # avoid forced resize to multiples of 16
    )

    fig, ax, scat_points, scat_medians = setup_figure()
    fig.canvas.draw()

    dbscan = DBSCAN(eps=2.5, min_samples=650, n_jobs=-1)
    medians_previous_frame = {}
    id_colors = {}
    colors = ["#FF0000", "#00FF00", "#FFFF00", "#00FFFF", "#FF00FF",
              "#C0C0C0", "#800000", "#808000", "#008000", "#800080", "#008080"]
    disappeared_medians = {}
    aabb_artists = {}
    prev_aabbs = {}

    try:
        for i, csv_name in enumerate(csv_files):
            csv_path = os.path.join(route, csv_name)
            print(f"Processing {i+1}/{len(csv_files)}: {csv_name}")
            df = pd.read_csv(csv_path)

            coords = df.select_dtypes(include=[np.number]).iloc[:, :3].to_numpy()
            dbscan.fit(coords)
            # get index of non-noise points

            noise = dbscan.labels_ == -1
            df = df[~noise]
            labels = dbscan.labels_[~noise]
            current_medians = compute_median_per_cluster(labels, df)

            if not medians_previous_frame:
                new_objects(np.unique(labels), id_colors, colors, labels)
            else:
                pairs, _, unmatchedB = pair_match_min_distance(medians_previous_frame, current_medians, max_distance=3)
                assign_matches(pairs, unmatchedB, labels, current_medians, id_colors, colors)
                track_disappeared(medians_previous_frame, current_medians, disappeared_medians, id_colors)

            self.plot_tracking(scat_points, df)

            raw_aabbs = compute_padded_aabbs_by_id(
                df, selfnew_labels,
                pad_rel=0.10,   # tweak: 0.05–0.20
                pad_abs=0.4,    # tweak in your units (meters, etc.)
                min_size=1.0,   # ensure a readable box
                min_pts=30
            )

            # optional smoothing to reduce frame-to-frame jitter
            aabbs = smooth_aabbs(prev_aabbs, raw_aabbs, alpha=0.6)
            # Make faces very transparent, keep edges thick and vivid
            update_aabbs(
                ax,
                aabbs, 
                id_colors, 
                aabb_artists,
                face_alpha=0.04,   # ↓ more transparent faces
                edge_alpha=0.98,   # bright edges
                edge_width=3.2     # same thickness you liked
            )


            prev_aabbs = aabbs


            fig.canvas.draw()

            h, w = fig.canvas.get_width_height()[1], fig.canvas.get_width_height()[0]
            rgba = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)
            rgb = rgba[..., :3]  # drop alpha for ffmpeg
            writer.append_data(rgb)

            medians_previous_frame = current_medians.copy()

    finally:
        writer.close()
        plt.close(fig)


In [ ]:
route = "/home/eder/projects/car-cluster-project/Data/pointclouds"
csv_sample = sorted(os.listdir(route))[:60]   # <- all your frame files
make_video(route, csv_sample, "/home/eder/projects/car-cluster-project/Eder/Video/cluster_tracking.mp4", fps=20)

In [483]:
class Tracking:
    def __init__(self, route, csv_files, out_path, fps=20):
        # Settings
        self.route = route
        self.csv_files = csv_files
        self.out_path = out_path
        self.fps = fps

        # Model
        self.dbscan = DBSCAN(eps=2.5, min_samples=650, n_jobs=-1)

        # Data
        self.id_counter = id() # Counter iterator for unique IDs
        self.colors = ["#FF0000", "#00FF00", "#FFFF00", "#00FFFF", "#FF00FF",
                       "#C0C0C0", "#800000", "#808000", "#008000", "#800080", "#008080"] # Colours used for different clusters, add more if needed
        self.id_colors = {} # Mapping from ID to color
        self.current_medians = {}  # {track_id: (x,y,z)}
        self.medians_previous_frame = {}  # {track_id: (x,y,z)}
        self.velocities = {} # {track_id: vx}
        self.disappeared_medians = {} # {track_id: count}
        self.labels = None  # To store labels for the current frame
        self.aabb_artists = {}
        self.prev_aabbs = {}

    def make_video(self, fps=20):
        # FFMPEG writer (no pyav)
        writer = imageio.get_writer(
            self.out_path,
            fps=fps,
            codec="libx264",
            format="FFMPEG",
            pixelformat="yuv420p",
            macro_block_size=None,  # avoid forced resize to multiples of 16
        )

        fig, ax, scat_points, scat_medians = self.setup_figure()
        fig.canvas.draw()

        try:
            for i, csv_name in enumerate(self.csv_files):
                csv_path = os.path.join(self.route, csv_name)
                print(f"Processing {i+1}/{len(self.csv_files)}: {csv_name}")
                df = pd.read_csv(csv_path)

                coords = df.select_dtypes(include=[np.number]).iloc[:, :3].to_numpy()
                self.dbscan.fit(coords)
                # get index of non-noise points

                noise = self.dbscan.labels_ == -1
                df = df[~noise]
                self.labels = self.dbscan.labels_[~noise]
                self.current_medians = self.compute_median_per_cluster(df)

                if not self.medians_previous_frame:
                    self.new_objects()
                else:
                    pairs, _, unmatchedB = self.pair_match_min_distance(max_distance=3)
                    self.assign_matches(pairs, unmatchedB)
                    self.track_disappeared()

                self.plot_tracking(scat_points, df)

                raw_aabbs = self.compute_padded_aabbs_by_id(
                    df,
                    pad_rel=0.10,   # tweak: 0.05–0.20
                    pad_abs=0.4,    # tweak in your units (meters, etc.)
                    min_size=1.0,   # ensure a readable box
                    min_pts=30
                )

                # optional smoothing to reduce frame-to-frame jitter
                aabbs = self.smooth_aabbs(self.prev_aabbs, raw_aabbs, alpha=0.6)
                # Make faces very transparent, keep edges thick and vivid
                self.update_aabbs(
                    ax,
                    aabbs,
                    face_alpha=0.04,   # ↓ more transparent faces
                    edge_alpha=0.98,   # bright edges
                    edge_width=3.2     # same thickness you liked
                )

                self.prev_aabbs = aabbs

                fig.canvas.draw()

                h, w = fig.canvas.get_width_height()[1], fig.canvas.get_width_height()[0]
                rgba = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)
                rgb = rgba[..., :3]  # drop alpha for ffmpeg
                writer.append_data(rgb)

                self.medians_previous_frame = self.current_medians.copy()

        finally:
            writer.close()
            plt.close(fig)

    def compute_median_per_cluster(self, df: pd.DataFrame):
        med = {}
        # Work on a temporary frame; do not modify df
        tmp = pd.DataFrame({'cluster': self.labels, 'x': df['x'].to_numpy(),
                            'y': df['y'].to_numpy(), 'z': df['z'].to_numpy()})
        tmp = tmp[tmp['cluster'] != -1]
        if tmp.empty:
            return med
        g = tmp.groupby('cluster', as_index=False)[['x','y','z']].median()
        for _, row in g.iterrows():
            med[int(row['cluster'])] = (float(row['x']), float(row['y']), float(row['z']))
        return med

    def assign_matches(self, pairs, unmatchedB):
        good_current_medians = {}
        good_labels = self.labels.copy()

        # Assign matched clusters
        for j, k, _ in pairs:
            good_labels[self.labels == k] = j
            good_current_medians[j] = self.current_medians.pop(k)

        # Assign new IDs to unmatched clusters
        for cluster in unmatchedB:
            id = next(self.id_counter)
            good_labels[self.labels ==cluster] = id
            self.id_colors[id] = self.colors[id]
            good_current_medians[id] = self.current_medians[cluster]

        # Set the updated data
        self.labels = good_labels
        self.current_medians = good_current_medians

    # Assign new IDs to new objects for the first iteration
    def new_objects(self):
        for cluster in np.unique(self.labels):
            id = next(self.id_counter)
            self.labels[cluster] = id
            self.id_colors[id] = self.colors[id]
    
    def track_disappeared(self):
        for k in self.medians_previous_frame.keys():
            if k not in self.current_medians:
                if k not in self.disappeared_medians:
                    self.disappeared_medians[k] = 1
                    self.current_medians[k] = self.medians_previous_frame[k]
                else:
                    self.disappeared_medians[k] = self.disappeared_medians.get(k, 0) + 1
                    if self.disappeared_medians[k] > 4:
                        self.id_colors.pop(k, None)
                        self.disappeared_medians.pop(k, None)
                    else:
                        self.current_medians[k] = self.medians_previous_frame[k]

    # Counter iterator
    def id(self):
        i = 0
        while True:
            yield i
            i += 1

    def plot_state(self):
        fig = plt.figure(figsize=(10, 5), dpi=120)
        ax = fig.add_subplot(111, projection='3d')
        ax.set_xlim([-70, 20])
        ax.set_ylim([-20, 20])
        ax.set_zlim([-20, 20])
        ax.set_box_aspect([4, 1, 1])  # Different aspect ratio
        ax.view_init(elev=10., azim=120)
        
        # base artists we’ll update
        scat_points = ax.scatter([], [], [], s=3)           # all points
        scat_medians = ax.scatter([], [], [], s=40, marker='x')  # cluster medians
        return fig, ax, scat_points, scat_medians

    def pair_match_min_distance(
        self,
        max_distance: float | None = None,
        allow_unmatched: bool = True,
        skip_cost: float | None = None,
    ):
        """
        Hungarian match between dicts of 3D points.
        Returns:
            pairs: [(keyA, keyB, distance)]
            unmatched_A: [keyA]
            unmatched_B: [keyB]
        """

        keysA = list(self.medians_previous_frame.keys())
        keysB = list(self.current_medians.keys())

        ArrA = np.asarray([self.medians_previous_frame[k] for k in keysA], dtype=float)  # (n,3)
        ArrB = np.asarray([self.current_medians[k] for k in keysB], dtype=float)  # (m,3)

        # Pairwise distances with broadcasting
        diff = ArrA[:, None, :] - ArrB[None, :, :]             # (n,m,3)
        D = np.linalg.norm(diff, axis=2)                       # (n,m)

        # Replace invalids with a huge cost
        BIG = 1e12
        finite_mask = np.isfinite(D)
        if not finite_mask.any():
            # nothing finite -> cannot match
            return [], keysA, keysB
        max_finite = D[finite_mask].max()
        D = np.where(finite_mask, D, BIG)

        # If we do not allow unmatched, do rectangular assignment
        if not allow_unmatched:
            r, c = linear_sum_assignment(D)
            pairs = [(keysA[i], keysB[j], float(D[i, j])) for i, j in zip(r, c)]
            unmatched_A = [keysA[i] for i in sorted(set(range(D.shape[0])) - set(r))]
            unmatched_B = [keysB[j] for j in sorted(set(range(D.shape[1])) - set(c))]
            return pairs, unmatched_A, unmatched_B

        # Allow unmatched: build square cost matrix with dummies
        pad = (max_finite + 1.0) if (skip_cost is None) else float(skip_cost)
        # ensure pad > any real distance so real matches are preferred
        if pad <= max_finite:
            pad = max_finite + 1.0

        n, m = D.shape
        size = n + m
        C = np.full((size, size), BIG, dtype=float)
        # real distances
        C[:n, :m] = D
        # A -> dummy columns (leave A[i] unmatched)
        C[:n, m:m+n] = pad
        # dummy rows -> B (leave B[j] unmatched)
        C[n:n+m, :m] = pad
        # bottom-right remains BIG (dummy-dummy forbidden)

        r, c = linear_sum_assignment(C)

        pairs_idx: List[Tuple[int, int]] = []
        unmatched_A_idx = set(range(n))
        unmatched_B_idx = set(range(m))

        for i, j in zip(r, c):
            if i < n and j < m:
                pairs_idx.append((i, j))
                unmatched_A_idx.discard(i)
                unmatched_B_idx.discard(j)
            # other cases are matches to dummies → remain unmatched

        # Fallback: if somehow all matched to dummies, try rectangular
        if not pairs_idx and (n > 0 and m > 0):
            rr, cc = linear_sum_assignment(D)
            pairs_idx = list(zip(rr, cc))
            unmatched_A_idx = set(range(n)) - set(rr)
            unmatched_B_idx = set(range(m)) - set(cc)

        pairs = [(keysA[i], keysB[j], float(D[i, j])) for i, j in pairs_idx]
        unmatched_A = [keysA[i] for i in sorted(unmatched_A_idx)]
        unmatched_B = [keysB[j] for j in sorted(unmatched_B_idx)]

        # Remove pairs with big distance (invalid matches)
        valid_pairs = []
        for (ka, kb, dist) in pairs:
            if max_distance > dist:
                valid_pairs.append((ka, kb, dist))
            else:
                unmatched_A.append(ka)
                unmatched_B.append(kb)
        pairs = valid_pairs
        return pairs, unmatched_A, unmatched_B
    

    def setup_figure(self):
        fig = plt.figure(figsize=(16, 8), dpi=200)
        ax = fig.add_subplot(111, projection='3d')
        ax.set_xlim([-70, 10])
        ax.set_ylim([-5, 5])
        ax.set_zlim([-5, 5])
        ax.set_box_aspect([4, 1, 1])
        ax.view_init(elev=30., azim=-90)

        #distance (overall zoom)
        ax.dist = 5  # default ~10; smaller -> closer, larger -> farther

        # base artists we’ll update
        scat_points = ax.scatter([], [], [], s=3)           # all points
        scat_medians = ax.scatter([], [], [], s=40, marker='x')  # cluster medians
        return fig, ax, scat_points, scat_medians

    def _rgba(self, color, alpha):
        r,g,b,_ = mcolors.to_rgba(color)
        return (r, g, b, float(alpha))

    def compute_padded_aabbs_by_id(self,
        df: pd.DataFrame,
        pad_rel: float = 0.10,     # 10% of each extent
        pad_abs: float = 0.5,      # or at least 0.5 units padding
        min_size: float = 1.0,     # ensure box not thinner than this (per axis)
        min_pts: int = 10          # ignore tiny clusters
    ):
        """
        Return {track_id: ((xmin,xmax),(ymin,ymax),(zmin,zmax))} with padding.
        """
        out = {}
        arr = df[['x','y','z']].to_numpy()
        for tid in np.unique(self.labels):
            mask = (self.labels == tid)
            if np.count_nonzero(mask) < min_pts:
                continue
            pts = arr[mask]
            mins = pts.min(axis=0)  # [xmin,ymin,zmin]
            maxs = pts.max(axis=0)  # [xmax,ymax,zmax]
            span = np.maximum(maxs - mins, 1e-6)

            # padding per-axis: max(relative, absolute)
            pad = np.maximum(pad_rel * span, pad_abs)

            # enforce minimum size (center the expansion)
            desired = np.maximum(span + 2*pad, min_size)
            center = (mins + maxs) * 0.5
            half = desired * 0.5

            bb_min = center - half
            bb_max = center + half
            out[int(tid)] = ((bb_min[0], bb_max[0]), (bb_min[1], bb_max[1]), (bb_min[2], bb_max[2]))
        return out

    def smooth_aabbs(self, prev: dict, curr: dict, alpha: float = 0.6):
        """
        Exponential smoothing of boxes to reduce flicker:
        new = alpha*curr + (1-alpha)*prev  (on mins/maxs separately)
        """
        smoothed = {}
        for tid, bb in curr.items():
            if tid in prev:
                (xmin,xmax),(ymin,ymax),(zmin,zmax) = bb
                (pxmin,pxmax),(pymin,pymax),(pzmin,pzmax) = prev[tid]
                sxmin = alpha*xmin + (1-alpha)*pxmin
                sxmax = alpha*xmax + (1-alpha)*pxmax
                symin = alpha*ymin + (1-alpha)*pymin
                symax = alpha*ymax + (1-alpha)*pymax
                szmin = alpha*zmin + (1-alpha)*pzmin
                szmax = alpha*zmax + (1-alpha)*pzmax
                smoothed[tid] = ((sxmin,sxmax),(symin,symax),(szmin,szmax))
            else:
                smoothed[tid] = bb
        return smoothed

    def _aabb_faces(self, bbox):
        """Return 6 quad faces from ((xmin,xmax),(ymin,ymax),(zmin,zmax))."""
        (xmin,xmax),(ymin,ymax),(zmin,zmax) = bbox
        c = np.array([
            [xmin,ymin,zmin],[xmax,ymin,zmin],[xmax,ymax,zmin],[xmin,ymax,zmin],
            [xmin,ymin,zmax],[xmax,ymin,zmax],[xmax,ymax,zmax],[xmin,ymax,zmax],
        ])
        F = [
            [0,1,2,3], [4,5,6,7],     # bottom, top
            [0,1,5,4], [1,2,6,5],     # sides
            [2,3,7,6], [3,0,4,7],
        ]
        return [c[idx] for idx in F]

    def update_aabbs(
        self,
        ax,
        aabbs_by_id: dict,         # {tid: ((xmin,xmax),...)}
        face_alpha: float = 0.12,
        edge_alpha: float = 0.95,
        edge_width: float = 3.0
    ):
        # remove stale
        for tid in list(self.aabb_artists.keys()):
            if tid not in aabbs_by_id:
                self.aabb_artists[tid].remove()
                del self.aabb_artists[tid]
        # upsert
        for tid, bb in aabbs_by_id.items():
            faces = _aabb_faces(bb)
            base = self.id_colors.get(int(tid), "#000000")
            fc = _rgba(base, face_alpha)
            ec = _rgba(base, edge_alpha)
            if tid in self.aabb_artists:
                poly = self.aabb_artists[tid]
                poly.set_verts(faces)
                poly.set_facecolor(fc)
                poly.set_edgecolor(ec)
                poly.set_linewidth(edge_width)
            else:
                poly = Poly3DCollection(faces, facecolors=fc, edgecolors=ec, linewidths=edge_width)
                try: poly.set_depthshade(False)
                except Exception: pass
                ax.add_collection3d(poly)
                self.aabb_artists[tid] = poly


    def plot_tracking(
        self,
        scat_points,
        df: pd.DataFrame,
    ):

        # ---- coords ----
        arr = df.select_dtypes(include=[np.number]).iloc[:, :3].to_numpy()
        x, y, z = arr[:, 0], arr[:, 1], arr[:, 2]

        # ---- mask out noise (keep arrays aligned) ----
        point_colors = np.empty((len(self.labels), 4), dtype=float)
        for id in self.current_medians.keys():
            color = mcolors.to_rgba(self.id_colors.get(int(id)))
            point_colors[self.labels == id] = color

        # ---- update point scatter ----
        scat_points._offsets3d = (x, y, z)
        scat_points.set_facecolors(point_colors)
        scat_points.set_edgecolors(point_colors)

    def plot_medians(
        self,
        scat_medians,
        median_size: float = 300.0,           # points^2 (Matplotlib convention)
        median_marker_edgewidth: float = 2.0, # thicker edges
        use_facecolor: bool = False,          # True if you use marker='o'
    ):
        if not self.current_medians:
            scat_medians._offsets3d = ([], [], [])
            scat_medians.set_sizes([])
            return

        # Keep order stable
        tids = list(self.current_medians.keys())
        meds = np.asarray([self.current_medians[t] for t in tids], dtype=float)
        mx, my, mz = meds[:, 0], meds[:, 1], meds[:, 2]

        # Update positions
        scat_medians._offsets3d = (mx, my, mz)

        # Set sizes (one per point)
        sizes = np.full(len(tids), float(median_size), dtype=float)
        scat_medians.set_sizes(sizes)

        # Colors: for marker='x', edgecolors matter; facecolors are ignored
        edge_cols = [mcolors.to_rgba(self.id_colors.get(int(t), "#000000")) for t in tids]
        scat_medians.set_edgecolors(edge_cols)
        scat_medians.set_linewidths(median_marker_edgewidth)

        # If your marker is 'o' (filled), use facecolors:
        if use_facecolor:
            scat_medians.set_facecolors(edge_cols)
        else:
            # For 'x', keep face fully transparent (but NOT the edges!)
            scat_medians.set_facecolors([(0, 0, 0, 0)] * len(tids))

        # Optional: avoid depth shading dimming the color
        try:
            scat_medians.set_depthshade(False)
        except Exception:
            pass
    

In [484]:
route = "/home/eder/projects/car-cluster-project/Data/pointclouds"
csv_sample = sorted(os.listdir(route))   # <- all your frame files

# Tracking
tracking = Tracking(route, csv_sample, "/home/eder/projects/car-cluster-project/Eder/Video/cluster_tracking_class.mp4", fps=20)
tracking.make_video()

Processing 1/369: pointcloud_1727346186_488981170.csv
Processing 2/369: pointcloud_1727346186_538003836.csv
Processing 3/369: pointcloud_1727346186_586942416.csv
Processing 4/369: pointcloud_1727346186_638212608.csv
Processing 5/369: pointcloud_1727346186_688448845.csv
Processing 6/369: pointcloud_1727346186_738318465.csv
Processing 7/369: pointcloud_1727346186_789278730.csv
Processing 8/369: pointcloud_1727346186_838301348.csv
Processing 9/369: pointcloud_1727346186_887269654.csv
Processing 10/369: pointcloud_1727346186_936946076.csv
Processing 11/369: pointcloud_1727346186_987057480.csv
Processing 12/369: pointcloud_1727346187_138756258.csv
Processing 13/369: pointcloud_1727346187_188181141.csv
Processing 14/369: pointcloud_1727346187_237884728.csv
Processing 15/369: pointcloud_1727346187_288974464.csv
Processing 16/369: pointcloud_1727346187_336883338.csv
Processing 17/369: pointcloud_1727346187_386856659.csv
Processing 18/369: pointcloud_1727346187_39143223.csv
Processing 19/369: p

In [486]:
class TrackingVelocity:
    def __init__(self, route, csv_files, out_path, fps=20):
        # Settings
        self.route = route
        self.csv_files = csv_files
        self.out_path = out_path
        self.fps = fps

        # Model
        self.dbscan = DBSCAN(eps=2.5, min_samples=650, n_jobs=-1)

        # Data
        self.id_counter = id() # Counter iterator for unique IDs
        self.colors = ["#FF0000", "#00FF00", "#FFFF00", "#00FFFF", "#FF00FF",
                       "#C0C0C0", "#800000", "#808000", "#008000", "#800080", "#008080"] # Colours used for different clusters, add more if needed
        self.id_colors = {} # Mapping from ID to color
        self.current_medians = {}  # {track_id: (x,y,z)}
        self.medians_previous_frame = {}  # {track_id: (x,y,z)}
        self.velocities = {} # {track_id: vx}
        self.disappeared_medians = {} # {track_id: count}
        self.labels = None  # To store labels for the current frame
        self.aabb_artists = {}
        self.prev_aabbs = {}

    def make_video(self, fps=20):
        # FFMPEG writer (no pyav)
        writer = imageio.get_writer(
            self.out_path,
            fps=fps,
            codec="libx264",
            format="FFMPEG",
            pixelformat="yuv420p",
            macro_block_size=None,  # avoid forced resize to multiples of 16
        )

        fig, ax, scat_points, scat_medians = self.setup_figure()
        fig.canvas.draw()

        try:
            for i, csv_name in enumerate(self.csv_files):
                csv_path = os.path.join(self.route, csv_name)
                print(f"Processing {i+1}/{len(self.csv_files)}: {csv_name}")
                df = pd.read_csv(csv_path)

                coords = df.select_dtypes(include=[np.number]).iloc[:, :3].to_numpy()
                self.dbscan.fit(coords)
                # get index of non-noise points

                noise = self.dbscan.labels_ == -1
                df = df[~noise]
                self.labels = self.dbscan.labels_[~noise]
                self.current_medians = self.compute_median_per_cluster(df)

                if not self.medians_previous_frame:
                    self.new_objects()
                else:
                    self.calculate_velocity()
                    pairs, _, unmatchedB = self.pair_match_min_distance(max_distance=3)
                    self.assign_matches(pairs, unmatchedB)
                    self.track_disappeared()

                self.plot_tracking(scat_points, df)

                raw_aabbs = self.compute_padded_aabbs_by_id(
                    df,
                    pad_rel=0.10,   # tweak: 0.05–0.20
                    pad_abs=0.4,    # tweak in your units (meters, etc.)
                    min_size=1.0,   # ensure a readable box
                    min_pts=30
                )

                # optional smoothing to reduce frame-to-frame jitter
                aabbs = self.smooth_aabbs(self.prev_aabbs, raw_aabbs, alpha=0.6)
                # Make faces very transparent, keep edges thick and vivid
                self.update_aabbs(
                    ax,
                    aabbs,
                    face_alpha=0.04,   # ↓ more transparent faces
                    edge_alpha=0.98,   # bright edges
                    edge_width=3.2     # same thickness you liked
                )

                self.prev_aabbs = aabbs

                fig.canvas.draw()

                h, w = fig.canvas.get_width_height()[1], fig.canvas.get_width_height()[0]
                rgba = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)
                rgb = rgba[..., :3]  # drop alpha for ffmpeg
                writer.append_data(rgb)

                self.medians_previous_frame = self.current_medians.copy()

        finally:
            writer.close()
            plt.close(fig)

    def compute_median_per_cluster(self, df: pd.DataFrame):
        med = {}
        # Work on a temporary frame; do not modify df
        tmp = pd.DataFrame({'cluster': self.labels, 'x': df['x'].to_numpy(),
                            'y': df['y'].to_numpy(), 'z': df['z'].to_numpy()})
        tmp = tmp[tmp['cluster'] != -1]
        if tmp.empty:
            return med
        g = tmp.groupby('cluster', as_index=False)[['x','y','z']].median()
        for _, row in g.iterrows():
            med[int(row['cluster'])] = (float(row['x']), float(row['y']), float(row['z']))
        return med

    def assign_matches(self, pairs, unmatchedB):
        good_current_medians = {}
        good_labels = self.labels.copy()

        # Assign matched clusters
        for j, k, _ in pairs:
            good_labels[self.labels == k] = j
            good_current_medians[j] = self.current_medians.pop(k)

        # Assign new IDs to unmatched clusters
        for cluster in unmatchedB:
            id = next(self.id_counter)
            good_labels[self.labels ==cluster] = id
            self.id_colors[id] = self.colors[id]
            good_current_medians[id] = self.current_medians[cluster]

        # Set the updated data
        self.labels = good_labels
        self.current_medians = good_current_medians

    def calculate_velocity(self):
        for id, (x, _, _) in self.current_medians.items():
            if id in self.medians_previous_frame:
                x_prev, _, _ = self.medians_previous_frame[id]
                vx = (x - x_prev) / (1 / self.fps)   # Assuming time delta is 1 unit
                self.velocities[id] = vx

    # Assign new IDs to new objects for the first iteration
    def new_objects(self):
        for cluster in np.unique(self.labels):
            id = next(self.id_counter)
            self.labels[cluster] = id
            self.id_colors[id] = self.colors[id]
            self.velocities[id] = 0.0  # Initialize velocity to zero
    
    def track_disappeared(self):
        for k in self.medians_previous_frame.keys():
            if k not in self.current_medians:
                if k not in self.disappeared_medians:
                    self.disappeared_medians[k] = 1
                    self.current_medians[k] = self.medians_previous_frame[k]
                else:
                    self.disappeared_medians[k] = self.disappeared_medians.get(k, 0) + 1
                    if self.disappeared_medians[k] > 4:
                        self.id_colors.pop(k, None)
                        self.disappeared_medians.pop(k, None)
                    else:
                        self.current_medians[k] = self.medians_previous_frame[k]

    # Counter iterator
    def id(self):
        i = 0
        while True:
            yield i
            i += 1

    def plot_state(self):
        fig = plt.figure(figsize=(10, 5), dpi=120)
        ax = fig.add_subplot(111, projection='3d')
        ax.set_xlim([-70, 20])
        ax.set_ylim([-20, 20])
        ax.set_zlim([-20, 20])
        ax.set_box_aspect([4, 1, 1])  # Different aspect ratio
        ax.view_init(elev=10., azim=120)
        
        # base artists we’ll update
        scat_points = ax.scatter([], [], [], s=3)           # all points
        scat_medians = ax.scatter([], [], [], s=40, marker='x')  # cluster medians
        return fig, ax, scat_points, scat_medians

    def pair_match_min_distance(
        self,
        max_distance: float | None = None,
        allow_unmatched: bool = True,
        skip_cost: float | None = None,
    ):
        """
        Hungarian match between dicts of 3D points.
        Returns:
            pairs: [(keyA, keyB, distance)]
            unmatched_A: [keyA]
            unmatched_B: [keyB]
        """
        # Apply velocity prediction to previous medians
        for id, (x, y, z) in self.medians_previous_frame.items():
            if id in self.velocities:
                vx = self.velocities[id]
                self.medians_previous_frame[id] = (x + vx, y, z)

        keysA = list(self.medians_previous_frame.keys())
        keysB = list(self.current_medians.keys())

        ArrA = np.asarray([self.medians_previous_frame[k] for k in keysA], dtype=float)  # (n,3)
        ArrB = np.asarray([self.current_medians[k] for k in keysB], dtype=float)  # (m,3)

        # Pairwise distances with broadcasting
        diff = ArrA[:, None, :] - ArrB[None, :, :]             # (n,m,3)
        D = np.linalg.norm(diff, axis=2)                       # (n,m)

        # Replace invalids with a huge cost
        BIG = 1e12
        finite_mask = np.isfinite(D)
        if not finite_mask.any():
            # nothing finite -> cannot match
            return [], keysA, keysB
        max_finite = D[finite_mask].max()
        D = np.where(finite_mask, D, BIG)

        # If we do not allow unmatched, do rectangular assignment
        if not allow_unmatched:
            r, c = linear_sum_assignment(D)
            pairs = [(keysA[i], keysB[j], float(D[i, j])) for i, j in zip(r, c)]
            unmatched_A = [keysA[i] for i in sorted(set(range(D.shape[0])) - set(r))]
            unmatched_B = [keysB[j] for j in sorted(set(range(D.shape[1])) - set(c))]
            return pairs, unmatched_A, unmatched_B

        # Allow unmatched: build square cost matrix with dummies
        pad = (max_finite + 1.0) if (skip_cost is None) else float(skip_cost)
        # ensure pad > any real distance so real matches are preferred
        if pad <= max_finite:
            pad = max_finite + 1.0

        n, m = D.shape
        size = n + m
        C = np.full((size, size), BIG, dtype=float)
        # real distances
        C[:n, :m] = D
        # A -> dummy columns (leave A[i] unmatched)
        C[:n, m:m+n] = pad
        # dummy rows -> B (leave B[j] unmatched)
        C[n:n+m, :m] = pad
        # bottom-right remains BIG (dummy-dummy forbidden)

        r, c = linear_sum_assignment(C)

        pairs_idx: List[Tuple[int, int]] = []
        unmatched_A_idx = set(range(n))
        unmatched_B_idx = set(range(m))

        for i, j in zip(r, c):
            if i < n and j < m:
                pairs_idx.append((i, j))
                unmatched_A_idx.discard(i)
                unmatched_B_idx.discard(j)
            # other cases are matches to dummies → remain unmatched

        # Fallback: if somehow all matched to dummies, try rectangular
        if not pairs_idx and (n > 0 and m > 0):
            rr, cc = linear_sum_assignment(D)
            pairs_idx = list(zip(rr, cc))
            unmatched_A_idx = set(range(n)) - set(rr)
            unmatched_B_idx = set(range(m)) - set(cc)

        pairs = [(keysA[i], keysB[j], float(D[i, j])) for i, j in pairs_idx]
        unmatched_A = [keysA[i] for i in sorted(unmatched_A_idx)]
        unmatched_B = [keysB[j] for j in sorted(unmatched_B_idx)]

        # Remove pairs with big distance (invalid matches)
        valid_pairs = []
        for (ka, kb, dist) in pairs:
            if max_distance > dist:
                valid_pairs.append((ka, kb, dist))
            else:
                unmatched_A.append(ka)
                unmatched_B.append(kb)
        pairs = valid_pairs
        return pairs, unmatched_A, unmatched_B
    

    def setup_figure(self):
        fig = plt.figure(figsize=(16, 8), dpi=200)
        ax = fig.add_subplot(111, projection='3d')
        ax.set_xlim([-70, 20])
        ax.set_ylim([-20, 20])
        ax.set_zlim([-20, 20])
        ax.set_box_aspect([4, 1, 1])
        ax.view_init(elev=30., azim=-90)

        #distance (overall zoom)
        ax.dist = 5  # default ~10; smaller -> closer, larger -> farther

        # base artists we’ll update
        scat_points = ax.scatter([], [], [], s=3)           # all points
        scat_medians = ax.scatter([], [], [], s=40, marker='x')  # cluster medians
        return fig, ax, scat_points, scat_medians

    def _rgba(self, color, alpha):
        r,g,b,_ = mcolors.to_rgba(color)
        return (r, g, b, float(alpha))

    def compute_padded_aabbs_by_id(self,
        df: pd.DataFrame,
        pad_rel: float = 0.10,     # 10% of each extent
        pad_abs: float = 0.5,      # or at least 0.5 units padding
        min_size: float = 1.0,     # ensure box not thinner than this (per axis)
        min_pts: int = 10          # ignore tiny clusters
    ):
        """
        Return {track_id: ((xmin,xmax),(ymin,ymax),(zmin,zmax))} with padding.
        """
        out = {}
        arr = df[['x','y','z']].to_numpy()
        for tid in np.unique(self.labels):
            mask = (self.labels == tid)
            if np.count_nonzero(mask) < min_pts:
                continue
            pts = arr[mask]
            mins = pts.min(axis=0)  # [xmin,ymin,zmin]
            maxs = pts.max(axis=0)  # [xmax,ymax,zmax]
            span = np.maximum(maxs - mins, 1e-6)

            # padding per-axis: max(relative, absolute)
            pad = np.maximum(pad_rel * span, pad_abs)

            # enforce minimum size (center the expansion)
            desired = np.maximum(span + 2*pad, min_size)
            center = (mins + maxs) * 0.5
            half = desired * 0.5

            bb_min = center - half
            bb_max = center + half
            out[int(tid)] = ((bb_min[0], bb_max[0]), (bb_min[1], bb_max[1]), (bb_min[2], bb_max[2]))
        return out

    def smooth_aabbs(self, prev: dict, curr: dict, alpha: float = 0.6):
        """
        Exponential smoothing of boxes to reduce flicker:
        new = alpha*curr + (1-alpha)*prev  (on mins/maxs separately)
        """
        smoothed = {}
        for tid, bb in curr.items():
            if tid in prev:
                (xmin,xmax),(ymin,ymax),(zmin,zmax) = bb
                (pxmin,pxmax),(pymin,pymax),(pzmin,pzmax) = prev[tid]
                sxmin = alpha*xmin + (1-alpha)*pxmin
                sxmax = alpha*xmax + (1-alpha)*pxmax
                symin = alpha*ymin + (1-alpha)*pymin
                symax = alpha*ymax + (1-alpha)*pymax
                szmin = alpha*zmin + (1-alpha)*pzmin
                szmax = alpha*zmax + (1-alpha)*pzmax
                smoothed[tid] = ((sxmin,sxmax),(symin,symax),(szmin,szmax))
            else:
                smoothed[tid] = bb
        return smoothed

    def _aabb_faces(self, bbox):
        """Return 6 quad faces from ((xmin,xmax),(ymin,ymax),(zmin,zmax))."""
        (xmin,xmax),(ymin,ymax),(zmin,zmax) = bbox
        c = np.array([
            [xmin,ymin,zmin],[xmax,ymin,zmin],[xmax,ymax,zmin],[xmin,ymax,zmin],
            [xmin,ymin,zmax],[xmax,ymin,zmax],[xmax,ymax,zmax],[xmin,ymax,zmax],
        ])
        F = [
            [0,1,2,3], [4,5,6,7],     # bottom, top
            [0,1,5,4], [1,2,6,5],     # sides
            [2,3,7,6], [3,0,4,7],
        ]
        return [c[idx] for idx in F]

    def update_aabbs(
        self,
        ax,
        aabbs_by_id: dict,         # {tid: ((xmin,xmax),...)}
        face_alpha: float = 0.12,
        edge_alpha: float = 0.95,
        edge_width: float = 3.0
    ):
        # remove stale
        for tid in list(self.aabb_artists.keys()):
            if tid not in aabbs_by_id:
                self.aabb_artists[tid].remove()
                del self.aabb_artists[tid]
        # upsert
        for tid, bb in aabbs_by_id.items():
            faces = _aabb_faces(bb)
            base = self.id_colors.get(int(tid), "#000000")
            fc = _rgba(base, face_alpha)
            ec = _rgba(base, edge_alpha)
            if tid in self.aabb_artists:
                poly = self.aabb_artists[tid]
                poly.set_verts(faces)
                poly.set_facecolor(fc)
                poly.set_edgecolor(ec)
                poly.set_linewidth(edge_width)
            else:
                poly = Poly3DCollection(faces, facecolors=fc, edgecolors=ec, linewidths=edge_width)
                try: poly.set_depthshade(False)
                except Exception: pass
                ax.add_collection3d(poly)
                self.aabb_artists[tid] = poly


    def plot_tracking(
        self,
        scat_points,
        df: pd.DataFrame,
    ):

        # ---- coords ----
        arr = df.select_dtypes(include=[np.number]).iloc[:, :3].to_numpy()
        x, y, z = arr[:, 0], arr[:, 1], arr[:, 2]

        # ---- mask out noise (keep arrays aligned) ----
        point_colors = np.empty((len(self.labels), 4), dtype=float)
        for id in self.current_medians.keys():
            color = mcolors.to_rgba(self.id_colors.get(int(id)))
            point_colors[self.labels == id] = color

        # ---- update point scatter ----
        scat_points._offsets3d = (x, y, z)
        scat_points.set_facecolors(point_colors)
        scat_points.set_edgecolors(point_colors)

    def plot_medians(
        self,
        scat_medians,
        median_size: float = 300.0,           # points^2 (Matplotlib convention)
        median_marker_edgewidth: float = 2.0, # thicker edges
        use_facecolor: bool = False,          # True if you use marker='o'
    ):
        if not self.current_medians:
            scat_medians._offsets3d = ([], [], [])
            scat_medians.set_sizes([])
            return

        # Keep order stable
        tids = list(self.current_medians.keys())
        meds = np.asarray([self.current_medians[t] for t in tids], dtype=float)
        mx, my, mz = meds[:, 0], meds[:, 1], meds[:, 2]

        # Update positions
        scat_medians._offsets3d = (mx, my, mz)

        # Set sizes (one per point)
        sizes = np.full(len(tids), float(median_size), dtype=float)
        scat_medians.set_sizes(sizes)

        # Colors: for marker='x', edgecolors matter; facecolors are ignored
        edge_cols = [mcolors.to_rgba(self.id_colors.get(int(t), "#000000")) for t in tids]
        scat_medians.set_edgecolors(edge_cols)
        scat_medians.set_linewidths(median_marker_edgewidth)

        # If your marker is 'o' (filled), use facecolors:
        if use_facecolor:
            scat_medians.set_facecolors(edge_cols)
        else:
            # For 'x', keep face fully transparent (but NOT the edges!)
            scat_medians.set_facecolors([(0, 0, 0, 0)] * len(tids))

        # Optional: avoid depth shading dimming the color
        try:
            scat_medians.set_depthshade(False)
        except Exception:
            pass
    

In [487]:
route = "/home/eder/projects/car-cluster-project/Data/pointclouds"
csv_sample = sorted(os.listdir(route))   # <- all your frame files

# Tracking
tracking = Tracking(route, csv_sample, "/home/eder/projects/car-cluster-project/Eder/Video/cluster_tracking_class_velocity.mp4", fps=20)
tracking.make_video()

Processing 1/369: pointcloud_1727346186_488981170.csv
Processing 2/369: pointcloud_1727346186_538003836.csv
Processing 3/369: pointcloud_1727346186_586942416.csv
Processing 4/369: pointcloud_1727346186_638212608.csv
Processing 5/369: pointcloud_1727346186_688448845.csv
Processing 6/369: pointcloud_1727346186_738318465.csv
Processing 7/369: pointcloud_1727346186_789278730.csv
Processing 8/369: pointcloud_1727346186_838301348.csv
Processing 9/369: pointcloud_1727346186_887269654.csv
Processing 10/369: pointcloud_1727346186_936946076.csv
Processing 11/369: pointcloud_1727346186_987057480.csv
Processing 12/369: pointcloud_1727346187_138756258.csv
Processing 13/369: pointcloud_1727346187_188181141.csv
Processing 14/369: pointcloud_1727346187_237884728.csv
Processing 15/369: pointcloud_1727346187_288974464.csv
Processing 16/369: pointcloud_1727346187_336883338.csv
Processing 17/369: pointcloud_1727346187_386856659.csv
Processing 18/369: pointcloud_1727346187_39143223.csv
Processing 19/369: p